# EMO1 - Catchment Time series
***

***Author:** Chus Casado Rodríguez*<br>
***Date:** 24-09-2026*<br>

**Introduction:**<br>

This notebook computes the catchment-aggregated meteorological timeseries for BEAVERS-ES for the European Meteorological Observations (EMO1):

* [EMO1](https://jeodpp.jrc.ec.europa.eu/ftp/jrc-opendata/CEMS-EFAS/meteorological_forcings/) provides daily or 6-hourly precipitation, temperature, wind speed, global radiation and vapour pressure. Most of these variables are used to compute reference potential evapotranspiration ($e_0$). I use the dataset provided by Gonçalo Gomes Ramos (JRC) that includes $e_0$, which is not in the link above. I'm using an older version of EMO1.

The time series start at January 1 1990 and end at December 31 2022. To speep up the computation of catchment-aggregations the EMO1 dataset was converted into Zarr format.

In [1]:
from tqdm.auto import tqdm

import geopandas as gpd
import xarray as xr
import logging
logger = logging.getLogger(__name__)

from ocab.config import Config
from ocab.basins.stats import read_data, read_pixarea, basin_statistics


## Configuration

In [2]:
# dataset configuration
cfg = Config('./config_BEAVERS_v100.yml')

# basins shapefile
basins_file = cfg.path_dataset / 'preprocessing' / 'basins' / 'output' / 'dams_basins_3sec.geojson'

# meteorology
meteo = 'EMO1'
cfg.path_meteo = cfg.path_meteo.parent.parent / meteo / 'Iberia'
zarr_store = cfg.path_meteo / f'{meteo}_1990-2022.zarr'

# output
path_out = cfg.path_dataset / 'preprocessing' / 'timeseries' / 'meteo' / meteo
overwrite = False
print(f'Output Parquet files will be saved in:\n{path_out}')

Output Parquet files will be saved in:
/home/casadoj/Data/BEAVERS-ES/v1_0_0/preprocessing/timeseries/meteo/EMO1


## Data

In [3]:
# load basins shapefile
if basins_file.is_file():
    basins = gpd.read_file(basins_file).set_index('ID')
else:
    logger.error(f"Basins file doesn't exist: {basins_file}")
print(f'No. basins: {len(basins)}')

No. basins: 366


In [4]:
# load meteorological data
if zarr_store.is_dir():
    data = read_data(zarr_store)
    print(f"{data.nbytes / 1e9:.2f} GB")
else:
    logger.error(f"Zarr store doesn't exist: {zarr_store}")

# rename variables
data = data.rename_vars({'pr': 'precipitation', 'ta': 'avgtemp'})

67.95 GB


In [5]:
# load pixel area to weigh the statistics
pixarea = read_pixarea(cfg.path_efas / 'maps' / 'pixarea_iberian_01min.nc')

## Basin Statistics

### Precipitation

In [6]:
# compute statistics
pcp_ts = basin_statistics(
    data=data['precipitation'],
    basins=basins,
    statistic=['mean', 'max', 'std'],
    weight=pixarea
)

# compute fraction area covered by precipitation
pcp_thr = 1 # mm
occurrence = xr.where(data['precipitation'] > pcp_thr, 1.0, 0.0)
occurrence = occurrence.rio.write_crs(data.rio.crs)
occurrence_ts = basin_statistics(
    data=occurrence,
    basins=basins,
    statistic=['count', 'sum'],
)
pcp_ts['precipitation_frac'] = occurrence_ts['precipitation_sum'] / occurrence_ts['precipitation_count']

basins:   0%|          | 0/366 [00:00<?, ?it/s]

2026-09-24 12:31:34,947 | INFO | ocab.basins.stats.main | Time elapsed: 283.58 seconds


basins:   0%|          | 0/366 [00:00<?, ?it/s]

2026-09-24 12:36:17,364 | INFO | ocab.basins.stats.main | Time elapsed: 282.23 seconds


### Temperature 

In [7]:
# compute statistics
vars = ['avgtemp', 'e0']
tmp_ts = basin_statistics(
    data=data[vars],
    basins=basins,
    statistic='mean',
    weight=pixarea
)

basins:   0%|          | 0/366 [00:00<?, ?it/s]

2026-09-24 12:49:26,135 | INFO | ocab.basins.stats.main | Time elapsed: 788.47 seconds


## Export

In [8]:
# combine variables
meteo_ts = xr.merge([pcp_ts, tmp_ts], compat='no_conflicts')

# export as Parquet files
path_out.mkdir(exist_ok=True)
for ID in tqdm(basins.index, desc='basins'):        
    fileout = path_out / f'{ID}.parquet'
    if fileout.exists() and  not overwrite:
        logger.info(f'Output file {fileout} already exists. Moving forward to the next basin')
        continue
    df = meteo_ts.sel(id=[ID]).to_dataframe()
    df.drop(['crs', 'spatial_ref', 'wgs_1984', 'rotated_pole', 'id'], axis=1, errors='ignore', inplace=True)
    df.to_parquet(fileout)

basins:   0%|          | 0/366 [00:00<?, ?it/s]